In [1]:
# [autoint_mlp_train.ipynb] 전체 실행 코드
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from tensorflow.keras.layers import Layer, Dense, Flatten, Dropout, BatchNormalization, Activation, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import TruncatedNormal, GlorotUniform
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. 뇌 청소
tf.keras.backend.clear_session()
print("🧠 메모리 초기화 완료")

# 2. 모델 클래스 정의 (로컬 VSCode 파일과 100% 일치하는 리스트 방식)
class FeaturesEmbedding(Layer):
    def __init__(self, field_dims, embed_dim, **kwargs):
        if 'name' not in kwargs: kwargs['name'] = 'fixed_embedding_layer'
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = Embedding(input_dim=self.total_dim, output_dim=self.embed_dim, name='emb_matrix')
    def build(self, input_shape):
        self.embedding.build(input_shape)
        self.embedding.set_weights([GlorotUniform()(shape=self.embedding.weights[0].shape)])
    def call(self, x):
        x = tf.cast(x, dtype=tf.int32)
        x = x + tf.constant(self.offsets)
        return self.embedding(x)

class MultiHeadSelfAttention(Layer):
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        super(MultiHeadSelfAttention, self).__init__(**kwargs)
        self.att_embedding_size, self.head_num, self.use_res, self.seed, self.scaling = att_embedding_size, head_num, use_res, seed, scaling
    def build(self, input_shape):
        embedding_size = int(input_shape[-1])
        self.W_Query = self.add_weight(name='query_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        self.W_key = self.add_weight(name='key_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 1))
        self.W_Value = self.add_weight(name='value_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed + 2))
        if self.use_res: self.W_Res = self.add_weight(name='res_W', shape=[embedding_size, self.att_embedding_size * self.head_num], dtype=tf.float32, initializer=TruncatedNormal(seed=self.seed))
        super(MultiHeadSelfAttention, self).build(input_shape)
    def call(self, inputs, **kwargs):
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))
        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))
        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling: inner_product /= self.att_embedding_size ** 0.5
        normalized_att_scores = tf.nn.softmax(inner_product)
        result = tf.matmul(normalized_att_scores, values)
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0)
        if self.use_res: result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        return tf.nn.relu(result)

class AutoIntMLP(Layer):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001, **kwargs):
        if 'name' not in kwargs: kwargs['name'] = 'fixed_autoint_mlp_layer'
        super(AutoIntMLP, self).__init__(**kwargs)
        self.embedding = FeaturesEmbedding(field_dims, embedding_size, name='fixed_embedding')
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size
        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_final_output')
        
        # [핵심] Sequential 제거 -> 리스트 사용
        self.dnn_layers = []
        for i, units in enumerate(dnn_hidden_units):
            self.dnn_layers.append(Dense(units, activation=None, kernel_regularizer=tf.keras.regularizers.l2(l2_reg_dnn), kernel_initializer=tf.random_normal_initializer(stddev=init_std), name=f'fixed_dnn_dense_{i}'))
            if dnn_use_bn: self.dnn_layers.append(BatchNormalization(name=f'fixed_dnn_bn_{i}'))
            self.dnn_layers.append(Activation(dnn_activation, name=f'fixed_dnn_act_{i}'))
            if dnn_dropout > 0: self.dnn_layers.append(Dropout(dnn_dropout, name=f'fixed_dnn_drop_{i}'))
        self.dnn_output_layer = Dense(1, kernel_initializer=tf.random_normal_initializer(stddev=init_std), name='fixed_dnn_output_layer')
        
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res, name=f'fixed_attention_{i}') for i in range(att_layer_num)]

    def call(self, inputs):
        embed_x = self.embedding(inputs)
        dnn_embed = tf.reshape(embed_x, shape=(-1, self.embedding_size * self.num_fields))
        att_input = embed_x
        for layer in self.int_layers: att_input = layer(att_input)
        att_output = Flatten()(att_input)
        att_output = self.final_layer(att_output)
        
        dnn_output = dnn_embed
        for layer in self.dnn_layers: dnn_output = layer(dnn_output)
        dnn_output = self.dnn_output_layer(dnn_output)
        return tf.keras.activations.sigmoid(att_output + dnn_output)

class AutoIntMLPModel(Model):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2,
                 att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu',
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False,
                 dnn_dropout=0.4, init_std=0.0001):
        super(AutoIntMLPModel, self).__init__(name='fixed_model_wrapper')
        self.autoInt_layer = AutoIntMLP(field_dims, embedding_size, att_layer_num, att_head_num, att_res, dnn_hidden_units, dnn_activation, l2_reg_dnn, l2_reg_embedding, dnn_use_bn, dnn_dropout, init_std)
    def call(self, inputs, training=False):
        return self.autoInt_layer(inputs, training=training)

# 3. 데이터 로드 및 학습
data_path = os.path.expanduser('~/aiffel/autoint/ml-1m')
movielens_rcmm = pd.read_csv(f"{data_path}/movielens_rcmm_v2.csv", dtype=str)
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]}
for col, le in label_encoders.items(): movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])
movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)
train_df, _ = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1

model = AutoIntMLPModel(field_dims=field_dims, embedding_size=16, att_layer_num=3, att_head_num=2, att_res=True, dnn_hidden_units=(32, 32), dnn_activation='relu', dnn_dropout=0.4)
model.compile(optimizer=Adam(learning_rate=0.0001), loss=BinaryCrossentropy(), metrics=['binary_crossentropy'])

print("🚀 학습 시작...")
model.fit(train_df[u_i_feature + meta_features], train_df['label'], epochs=3, batch_size=2048, validation_split=0.1)

# 4. 저장 및 검증
save_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.weights.h5')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
model.save_weights(save_path)
print(f"✅ 저장 완료: {save_path}")

# ★ 검증: 파일이 제대로 생성되었는지 확인
import h5py
try:
    with h5py.File(save_path, 'r') as f:
        print("🔍 저장된 파일 내부 구조 확인:")
        # 구조가 fixed_autoint_mlp_layer 바로 아래에 fixed_dnn_dense_0가 있어야 함 (중간에 stack 폴더가 없어야 함)
        if 'fixed_model_wrapper' in f and 'fixed_autoint_mlp_layer' in f['fixed_model_wrapper']:
            print("   -> OK: 모델 래퍼 발견")
            layer_grp = f['fixed_model_wrapper']['fixed_autoint_mlp_layer']
            if 'fixed_dnn_dense_0' in layer_grp:
                print("   -> 🎉 성공! 올바른 구조(List 방식)로 저장되었습니다!")
            else:
                print("   -> ⚠️ 주의: 아직도 Sequential 구조(stack)로 저장된 것 같습니다. 커널 리스타트를 다시 하세요.")
except Exception as e:
    print(f"⚠️ 파일 확인 중 에러 (무시 가능): {e}")

🧠 메모리 초기화 완료
🚀 학습 시작...
Epoch 1/3
352/352 [==============================] - 9s 16ms/step - loss: 0.6733 - binary_crossentropy: 0.6733 - val_loss: 0.6314 - val_binary_crossentropy: 0.6314
Epoch 2/3
352/352 [==============================] - 5s 14ms/step - loss: 0.5999 - binary_crossentropy: 0.5999 - val_loss: 0.5799 - val_binary_crossentropy: 0.5799
Epoch 3/3
352/352 [==============================] - 5s 14ms/step - loss: 0.5579 - binary_crossentropy: 0.5579 - val_loss: 0.5514 - val_binary_crossentropy: 0.5514
✅ 저장 완료: /aiffel/aiffel/autoint/model/autoInt_model_weights.weights.h5
🔍 저장된 파일 내부 구조 확인:


In [2]:
# [주피터 노트북] 가중치 추출 및 저장 (Pickle 방식)
import joblib
import os

# 1. 모델이 잘 학습되어 있는지 확인 (기존 학습 코드 실행 후)
weights_list = model.get_weights() # 모델의 모든 가중치를 리스트로 추출

# 2. 경로 설정
save_pkl_path = os.path.expanduser('~/aiffel/autoint/model/autoInt_model_weights.pkl')
os.makedirs(os.path.dirname(save_pkl_path), exist_ok=True)

# 3. 저장
joblib.dump(weights_list, save_pkl_path)
print(f"✅ [최종] 가중치 리스트 저장 완료: {save_pkl_path}")
print(f"   -> 총 {len(weights_list)}개의 가중치 덩어리가 저장되었습니다.")

✅ [최종] 가중치 리스트 저장 완료: /aiffel/aiffel/autoint/model/autoInt_model_weights.pkl
   -> 총 20개의 가중치 덩어리가 저장되었습니다.
